# EfficientViT-Seg-B1 segmentation on BTXRD (50 epochs)

Train binary tumor segmentation with an EfficientViT-Seg-B1 backbone using a Kaggle GPU. The same pinned BTXRD archive and `make_splits` filtering as the ViT-B/16 / Segmenter notebooks are used. Training runs for 50 epochs and saves `best_evit_seg_btxrd.pth` in `/kaggle/working/`. The test split is reserved for final evaluation with TTA.

In [9]:
import importlib.machinery
import os
import subprocess
import sys
import types
from pathlib import Path

assert Path('/kaggle/working').is_dir(), 'This notebook must run on Kaggle.'
os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TORCHDYNAMO_DISABLE'] = '1'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.31'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/mit-han-lab/efficientvit.git'], check=True)

try:
    import triton
    import triton.language
except Exception:
    def _make_module(name):
        module = types.ModuleType(name)
        module.__path__ = []
        module.__spec__ = importlib.machinery.ModuleSpec(name, None)
        sys.modules[name] = module
        return module

    _triton = _make_module('triton')
    _triton_language = _make_module('triton.language')
    _triton.backends = _make_module('triton.backends')
    _triton.backends.compiler = _make_module('triton.backends.compiler')
    _triton.compiler = _make_module('triton.compiler')
    _triton.compiler.compiler = _make_module('triton.compiler.compiler')
    _triton.language = _triton_language

    for _name, _value in {
        'jit': lambda fn=None, **kwargs: (fn if fn is not None else (lambda x: x)),
        'cdiv': lambda a, b: (a + b - 1) // b,
        'next_power_of_2': lambda x: 1 << (x - 1).bit_length(),
    }.items():
        setattr(_triton, _name, _value)
    for _name, _value in {
        'constexpr': object,
        'dtype': object,
        'program_id': lambda *args, **kwargs: 0,
        'arange': lambda *args, **kwargs: None,
        'zeros': lambda *args, **kwargs: None,
        'float32': float,
        'load': lambda *args, **kwargs: None,
        'store': lambda *args, **kwargs: None,
        'where': lambda *args, **kwargs: None,
        'sum': lambda *args, **kwargs: None,
    }.items():
        setattr(_triton_language, _name, _value)

import timm
import torch
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), 'CUDA GPU unavailable: check the Kaggle accelerator and GPU quota.'
print('Torch:', torch.__version__, 'timm:', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0), flush=True)

Torch: 2.11.0+cu130 timm: 1.0.31
GPU: Tesla T4


In [10]:
import hashlib

DATASET_REVISION = 'cf8b0569b18e43e7cecab412a06e60aeaf258b32'
EXPECTED_SHA256 = '59ce94244fdf997f0c68e6a378690a4359e6c411f7fda7ac2f3994489637884b'
archive = Path(hf_hub_download(repo_id='torusama/xla-seminar-uia', repo_type='dataset',
                               filename='btxrd_final.zip', revision=DATASET_REVISION))
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f'Dataset checksum mismatch: {digest}')
print('BTXRD downloaded; SHA-256:', digest, flush=True)


BTXRD downloaded; SHA-256: 59ce94244fdf997f0c68e6a378690a4359e6c411f7fda7ac2f3994489637884b


In [11]:
# Project sources are bundled in this notebook; no Git checkout is needed.
PROJECT = Path('/kaggle/temp/xla-efficientvit-seg')
SOURCES = {
    'src/xla/__init__.py': '"""Research scaffold for vision classification and segmentation."""\n\n__version__ = "0.1.0"\n',
    'src/xla/data/__init__.py': '"""Data interfaces."""\n',
    'src/xla/data/hf_segmentation.py': '"""Read the pinned Hugging Face archive and rasterize polygon masks."""\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\n\n\nclass PolygonArchive(Dataset):\n    """Binary union of all annotated polygons; normal samples have empty masks."""\n\n    def __init__(self, archive: Path, split: str, excluded_hashes: set[str] | None = None):\n        self.archive = Path(archive)\n        self.samples = []\n        self.excluded = 0\n        self._zip = None\n        self._zip_pid = None\n        excluded_hashes = excluded_hashes or set()\n        with ZipFile(self.archive) as z:\n            names = set(z.namelist())\n            annotations = sorted(n for n in names if n.startswith(f"{split}/annotations/")\n                                 and n.endswith(".json"))\n            for name in annotations:\n                annotation = json.loads(z.read(name))\n                category = Path(name).parent.name\n                image_name = f"{split}/images/{category}/{annotation[\'image_name\']}"\n                if image_name not in names:\n                    raise ValueError(f"Missing image for {name}: {image_name}")\n                digest = hashlib.sha256(z.read(image_name)).hexdigest()\n                if digest in excluded_hashes:\n                    self.excluded += 1\n                    continue\n                self.samples.append((image_name, name, digest))\n        if not self.samples:\n            raise ValueError(f"No samples in {self.archive} split {split}")\n\n    @property\n    def hashes(self) -> set[str]:\n        return {sample[2] for sample in self.samples}\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state["_zip"] = None\n        state["_zip_pid"] = None\n        return state\n\n    def __getitem__(self, index):\n        image_name, annotation_name, _ = self.samples[index]\n        if self._zip is None or self._zip_pid != os.getpid():\n            if self._zip is not None:\n                self._zip.close()\n            self._zip = ZipFile(self.archive)\n            self._zip_pid = os.getpid()\n        annotation = json.loads(self._zip.read(annotation_name))\n        with Image.open(io.BytesIO(self._zip.read(image_name))) as source:\n            image = source.convert("RGB")\n        expected = (annotation["image_width"], annotation["image_height"])\n        if image.size != expected or expected != (224, 224):\n            raise ValueError(f"Image and annotation dimensions do not match 224x224: {image_name}")\n        mask = Image.new("L", image.size, 0)\n        draw = ImageDraw.Draw(mask)\n        count = 0\n        for item in annotation["annotations"]:\n            for polygon in item.get("polygons", []):\n                if len(polygon) < 3:\n                    raise ValueError(f"Invalid polygon: {annotation_name}")\n                draw.polygon([tuple(point) for point in polygon], fill=1)\n                count += 1\n        if not count and annotation["class_label"].lower() != "normal":\n            raise ValueError(f"Non-normal sample has no polygons: {annotation_name}")\n        pixels = TF.normalize(TF.to_tensor(image), mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))\n        target = torch.frombuffer(bytearray(mask.tobytes()), dtype=torch.uint8).reshape(1, 224, 224)\n        return pixels, target.float()\n\n\ndef make_splits(archive: Path):\n    """Remove conflicting labels and train duplicates of held-out images."""\n    splits = [PolygonArchive(archive, name) for name in ("train", "val", "test")]\n    categories = {}\n    for split in splits:\n        for image_name, _, digest in split.samples:\n            categories.setdefault(digest, set()).add(Path(image_name).parent.name)\n    conflicts = {digest for digest, labels in categories.items() if len(labels) > 1}\n    for split in splits:\n        original = len(split.samples)\n        split.samples = [sample for sample in split.samples if sample[2] not in conflicts]\n        split.excluded = original - len(split.samples)\n    train, val, test = splits\n    if val.hashes & test.hashes:\n        raise ValueError("Validation and test contain identical images; resolve the split first")\n    held_out = val.hashes | test.hashes\n    original = len(train.samples)\n    train.samples = [sample for sample in train.samples if sample[2] not in held_out]\n    train.excluded += original - len(train.samples)\n    if any(not split.samples for split in splits):\n        raise ValueError("A split is empty after removing conflicting and duplicate images")\n    return train, val, test\n',
}
for name, content in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
print('Project sources prepared:', len(SOURCES), 'files', flush=True)


Project sources prepared: 3 files


In [12]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

import sys
import types
import importlib.util

# Chan triet de efficientvit.apps.utils.export de khong bao gio goi toi onnx hay ml_dtypes
_export_mod_name = "efficientvit.apps.utils.export"
if _export_mod_name not in sys.modules:
    _spec = importlib.util.spec_from_loader(_export_mod_name, loader=None)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.export_onnx = lambda *args, **kwargs: None
    sys.modules[_export_mod_name] = _mod

from efficientvit.seg_model_zoo import create_efficientvit_seg_model
from xla.data.hf_segmentation import make_splits

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda')
EPOCHS = 50
BATCH_SIZE = 8
LR = 2e-4
WEIGHT_DECAY = 1e-2
POS_WEIGHT = 10.0

train_set, val_set, test_set = make_splits(archive)
print(f"Samples: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}; "
      f"excluded: train={train_set.excluded}, val={val_set.excluded}, test={test_set.excluded}", flush=True)


class TrainAugmentDataset(Dataset):
    def __init__(self, base):
        self.base = base
        self.indices = [i for i in range(len(base)) if base[i][1].sum().item() > 0]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, mask = self.base[self.indices[idx]]
        if random.random() < 0.5:
            image = torch.flip(image, dims=[-1])
            mask = torch.flip(mask, dims=[-1])
        if random.random() < 0.5:
            angle = random.uniform(-15.0, 15.0)
            image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
        return image, mask


class SegDataset(Dataset):
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.base[idx]


train_dataset = TrainAugmentDataset(train_set)
val_dataset = SegDataset(val_set)
test_dataset = SegDataset(test_set)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=2)

print('Train tumor samples:', len(train_dataset), flush=True)


def load_backbone_from_timm(core):
    timm_model = timm.create_model('efficientvit_b1.r224_in1k', pretrained=True)
    timm_state = timm_model.state_dict()
    core_state = core.backbone.state_dict()
    filtered = [(key, value) for key, value in timm_state.items() if not key.startswith('head.')]
    assert len(filtered) == len(core_state), (len(filtered), len(core_state))
    mapped = {core_key: value for (core_key, _), (_, value) in zip(core_state.items(), filtered)}
    core.backbone.load_state_dict(mapped, strict=True)
    del timm_model


core = create_efficientvit_seg_model(name='efficientvit-seg-b1-ade20k', pretrained=False)
load_backbone_from_timm(core)

for output_op in core.head.output_ops:
    last_layer = output_op.op_list[-1]
    in_channels = last_layer.conv.in_channels
    last_layer.conv = nn.Conv2d(in_channels, 1, kernel_size=1, bias=True)


class EVITSeg(nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(self, images):
        logits = self.core(images)
        return F.interpolate(logits, size=images.shape[-2:], mode='bilinear', align_corners=False)


model = EVITSeg(core).to(device)
print('Parameters (M):', sum(p.numel() for p in model.parameters()) / 1e6, flush=True)


def segmentation_loss(logits, target):
    weight = torch.tensor(POS_WEIGHT, device=logits.device, dtype=logits.dtype)
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=weight)
    probabilities = logits.sigmoid()
    intersection = (probabilities * target).sum()
    denominator = probabilities.sum() + target.sum()
    dice_loss = 1.0 - (2.0 * intersection + 1.0) / (denominator + 1.0)
    return bce + dice_loss


def run_epoch(model, loader, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    tp = fp = fn = 0
    count = 0
    with torch.set_grad_enabled(training):
        for images, target in loader:
            images = images.to(device)
            target = target.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = segmentation_loss(logits, target)
            if training:
                loss.backward()
                optimizer.step()
            batch = images.shape[0]
            total_loss += loss.item() * batch
            count += batch
            prediction = logits.detach() >= 0
            truth = target.bool()
            tp += (prediction & truth).sum().item()
            fp += (prediction & ~truth).sum().item()
            fn += (~prediction & truth).sum().item()
    return {
        'loss': total_loss / count,
        'dice': (2 * tp / (2 * tp + fp + fn)) if (tp + fp + fn) else 1.0,
        'iou': (tp / (tp + fp + fn)) if (tp + fp + fn) else 1.0,
    }


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

output_dir = Path('/kaggle/working')
checkpoint_path = output_dir / 'best_evit_seg_btxrd.pth'
history = []
best_val_dice = -1.0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, device, optimizer)
    val_metrics = run_epoch(model, val_loader, device)
    scheduler.step()
    record = {'epoch': epoch, 'train': train_metrics, 'val': val_metrics, 'lr': optimizer.param_groups[0]['lr']}
    history.append(record)
    print(json.dumps(record), flush=True)
    (output_dir / 'evit_seg_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
    if val_metrics['dice'] > best_val_dice:
        best_val_dice = val_metrics['dice']
        torch.save(model.core.state_dict(), checkpoint_path)
        print('Saved best checkpoint:', checkpoint_path, 'val_dice=', val_metrics['dice'], flush=True)

assert checkpoint_path.is_file()
print('Finished training. Best val dice:', best_val_dice, flush=True)


ModuleNotFoundError: No module named 'xla'

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

checkpoint_path = Path('/kaggle/working/best_evit_seg_btxrd.pth')
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

core = create_efficientvit_seg_model(name='efficientvit-seg-b1-ade20k', pretrained=False)
# rebuild 1-channel head
for output_op in core.head.output_ops:
    last_layer = output_op.op_list[-1]
    in_channels = last_layer.conv.in_channels
    last_layer.conv = torch.nn.Conv2d(in_channels, 1, kernel_size=1, bias=True)
core.load_state_dict(checkpoint, strict=True)

class EVITSeg(torch.nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core
    def forward(self, images):
        logits = self.core(images)
        return torch.nn.functional.interpolate(logits, size=images.shape[-2:], mode='bilinear', align_corners=False)

model = EVITSeg(core).to(device)
model.eval()


def predict_tta(images):
    with torch.no_grad():
        prob_original = model(images).sigmoid()
        flipped_input = torch.flip(images, dims=[3])
        prob_flipped = torch.flip(model(flipped_input), dims=[3]).sigmoid()
    return (prob_original + prob_flipped) / 2.0

tp = fp = fn = 0
macro_dices = []
macro_ious = []
normal_total = 0
normal_clean = 0
inference_ms = []

with torch.no_grad():
    for images, target in test_loader:
        images = images.to(device)
        target = target.to(device)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        prob = predict_tta(images)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        inference_ms.append(elapsed / images.shape[0] * 1000.0)

        pred = prob >= 0.5
        truth = target.bool()

        tp += (pred & truth).sum().item()
        fp += (pred & ~truth).sum().item()
        fn += (~pred & truth).sum().item()

        for i in range(images.shape[0]):
            pred_i = pred[i]
            truth_i = truth[i]
            if target[i].sum().item() > 0:
                intersection = (pred_i & truth_i).sum().item()
                pred_pixels = pred_i.sum().item()
                truth_pixels = truth_i.sum().item()
                union = (pred_i | truth_i).sum().item()
                dice = (2.0 * intersection + 1e-6) / (pred_pixels + truth_pixels + 1e-6)
                iou = (intersection + 1e-6) / (union + 1e-6)
                macro_dices.append(dice)
                macro_ious.append(iou)
            else:
                normal_total += 1
                if pred[i].sum().item() < 25:
                    normal_clean += 1

global_dice = (2 * tp / (2 * tp + fp + fn)) if (tp + fp + fn) else 1.0
global_iou = (tp / (tp + fp + fn)) if (tp + fp + fn) else 1.0
mean_dice = float(np.mean(macro_dices)) if macro_dices else 1.0
mean_iou = float(np.mean(macro_ious)) if macro_ious else 1.0
specificity = (normal_clean / normal_total) if normal_total else 1.0

params_m = sum(p.numel() for p in model.parameters()) / 1e6
checkpoint_size_mb = checkpoint_path.stat().st_size / (1024 * 1024)
avg_inference_ms = float(np.mean(inference_ms))

metrics = {
    'global_micro_dice': global_dice,
    'global_micro_iou': global_iou,
    'mean_macro_dice_tumor': mean_dice,
    'mean_macro_iou_tumor': mean_iou,
    'specificity_normal': specificity,
    'parameters_m': params_m,
    'checkpoint_size_mb': checkpoint_size_mb,
    'avg_inference_ms_per_image': avg_inference_ms,
    'tumor_test_images': len(macro_dices),
    'normal_test_images': normal_total,
}
print(json.dumps(metrics, indent=2), flush=True)
Path('/kaggle/working/evit_seg_test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
